In [1]:
from openai import AzureOpenAI
import tiktoken
import json
import pandas as pd
import os

In [5]:
with open('../key.txt', "r", encoding="UTF-8") as file:
    key = file.read()
TEMPERATURE=0
MAX_TOKENS=6000
# azure_openai_key = ''
# azure_openai_endpoint = 'https://cursorplayground-002.openai.azure.com/'
# azure_api_verison = '2024-02-15-preview'
# azure_openai_model = 'cursor_35_16k' 
azure_openai_key = key
azure_openai_endpoint = 'https://ontology-kz.openai.azure.com/'
azure_api_verison = '2024-02-15-preview'
azure_openai_model = 'mCodegpt_gpt_4_32k' 

azure_endpoint = "https://cursorplayground-002.openai.azure.com/openai/deployments/cursor_35_16k/chat/completions?api-version=2024-02-15-preview"
azure_tpm= 120000
azure_rate_limit = 720
input_cost_per_million = 0.50
output_cost_per_million = 1.50

client = AzureOpenAI(api_key=azure_openai_key, azure_endpoint=azure_openai_endpoint, api_version=azure_api_verison)

In [6]:

input_dir = './input'
df = pd.DataFrame(columns=['File Name', 'Inclusion Criteria', 'Exclusion Criteria'])
abstract_list = [] 

for file_name in os.listdir(input_dir):
    if file_name.endswith('.txt'):
        file_path = os.path.join(input_dir, file_name)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            ### Split the notes to inclusion and exclusion criteria seperately
            inclusion_criteria = content.split('Exclusion Criteria:')[0].split('Inclusion Criteria:')[-1].strip()
            exclusion_criteria = content.split('Exclusion Criteria:')[-1].strip()
            new_row = pd.DataFrame({'File Name': [file_name], 'Inclusion Criteria': [inclusion_criteria], 'Exclusion Criteria': [exclusion_criteria]})
            df = pd.concat([df, new_row], ignore_index=True)

abstract_list = df[['File Name', 'Inclusion Criteria', 'Exclusion Criteria']].values.tolist()
print("Abstracts have been successfully read into a list.")
 

def process_abstracts_and_get_tool_arguments(abstract_list, client, tools):
    global TEMPERATURE
    global MAX_TOKENS
    tool_arguments_df = pd.DataFrame(columns=['File Name', 'Criteria Type', 'Arguments', 'Cost', 'Error'])
    processed_abstracts = []  

    for item in abstract_list:
        file_name, inclusion_criteria, exclusion_criteria = item
        for criteria_type, criteria_text in [('Inclusion', inclusion_criteria), ('Exclusion', exclusion_criteria)]:
            print("\n" + "="*50)
            print(f"Processing {criteria_type} criteria for file {file_name}...")
            print("="*50 + "\n")
            
            clean_text = criteria_text.replace("  ", " ").replace("\n", "; ").replace(';',' ')

            ### Improve the prompt with details
            prompt_template = '''Extract information from the clinical trial eligibility criteria. This is the f"Processing {criteria_type} criteria: <clean_text>.
            The output should follow the enumeration. If the enumeration is blank, and the criteria is not mentioned in the note, return 'not applicable'''
    
            prompt = prompt_template.replace("<clean_text>", clean_text)
        
            messages = [
              {"role": "system", "content": "You will help extract information from clinical protocol documents."},
              {"role": "user", "content": prompt}
              ]
            try:
                completion = client.chat.completions.create(
                    model= azure_openai_model,
                    messages=messages,
                    tools=tools,
                    tool_choice="auto",
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS
                    )
                tool_calls = completion.choices[0].message.tool_calls
                arguments_list = []
        
                for tool_call in tool_calls:
                    arguments_str = tool_call.function.arguments
                    arguments = json.loads(arguments_str)
                    arguments_list.append(arguments)
                prompt_tokens = completion.usage.prompt_tokens
                completion_tokens = completion.usage.completion_tokens
                total_tokens = completion.usage.total_tokens

                input_cost_per_token = input_cost_per_million / 1_000_000
                output_cost_per_token = output_cost_per_million / 1_000_000
                total_input_cost = prompt_tokens * input_cost_per_token
                total_output_cost = completion_tokens * output_cost_per_token
                total_cost = total_input_cost + total_output_cost
                print(f"Cost for processing {criteria_type} criteria for file {file_name}: {total_tokens} tokens, Total Cost: ${total_cost:.5f}")
                new_row = pd.DataFrame([{'File Name': file_name, 'Criteria Type': criteria_type, 'Arguments': arguments_list, 'Cost': total_cost, 'Error': None}])
                tool_arguments_df = pd.concat([tool_arguments_df, new_row], ignore_index=True)

                processed_abstracts.append({'file_name': file_name, 'criteria_type': criteria_type, 'abstract': criteria_text, 'arguments': arguments_list, 'cost': total_cost})

                with open('processed_abstracts.json', 'w') as f:
                    json.dump(processed_abstracts, f, indent=4)

            except Exception as e:
                print(f"An error occurred while processing {criteria_type} criteria for file {file_name}: {e}")
                new_row = pd.DataFrame([{'File Name': file_name, 'Criteria Type': criteria_type, 'Arguments': None, 'Cost': None, 'Error': str(e)}])
                tool_arguments_df = pd.concat([tool_arguments_df, new_row], ignore_index=True)
                processed_abstracts.append({'file_name': file_name, 'criteria_type': criteria_type, 'abstract': criteria_text, 'error': str(e)})
                with open('processed_abstracts.json', 'w') as f:
                    json.dump(processed_abstracts, f, indent=4)

    return tool_arguments_df


def generate_argument_entries(arguments):
    entries = []
    try:
        for arg in arguments:
            for key, value in arg.items():
                if isinstance(value, str) and value:
                    entry = f'"{key}": "{value}"'
                elif isinstance(value, (int, float)) and value != 0:
                    entry = f'"{key}": {value}'
                elif isinstance(value, str):
                    entry = f'"{key}": ""'
                else:
                    entry = f'"{key}": {value}'
                entries.append(entry)
    except:
        print("TypeError: 'NoneType' object is not iterable")
        return None
    return "\n".join(entries)

Abstracts have been successfully read into a list.


In [4]:
with open("GEARBOx.json", "r") as file:
    tool = json.load(file)
    
tools = [tool]
tool_arguments_df = process_abstracts_and_get_tool_arguments(abstract_list, client, tools)

from datetime import datetime
output_excel_path = f'output_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'

tool_arguments_df['Arguments Entries'] = tool_arguments_df['Arguments'].apply(generate_argument_entries)

output_df = tool_arguments_df[['File Name', 'Criteria Type', 'Arguments Entries', 'Cost', 'Error']]

output_df.to_excel(output_excel_path, index=False)
print(f"Data has been written to {output_excel_path}")


Processing Inclusion criteria for file NCT04898894.txt...

Cost for processing Inclusion criteria for file NCT04898894.txt: 6991 tokens, Total Cost: $0.00476

Processing Exclusion criteria for file NCT04898894.txt...



C:\Users\frakoto\AppData\Local\Temp\ipykernel_41756\421759107.py:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tool_arguments_df = pd.concat([tool_arguments_df, new_row], ignore_index=True)


Cost for processing Exclusion criteria for file NCT04898894.txt: 6301 tokens, Total Cost: $0.00441

Processing Inclusion criteria for file NCT04915612.txt...

Cost for processing Inclusion criteria for file NCT04915612.txt: 6670 tokens, Total Cost: $0.00460

Processing Exclusion criteria for file NCT04915612.txt...

Cost for processing Exclusion criteria for file NCT04915612.txt: 6703 tokens, Total Cost: $0.00461

Processing Inclusion criteria for file NCT05188170.txt...

Cost for processing Inclusion criteria for file NCT05188170.txt: 6694 tokens, Total Cost: $0.00461

Processing Exclusion criteria for file NCT05188170.txt...

Cost for processing Exclusion criteria for file NCT05188170.txt: 6482 tokens, Total Cost: $0.00450
Data has been written to output_data_20240730_153533.xlsx


In [6]:
tool_arguments_df['Arguments Entries'] = tool_arguments_df['Arguments'].apply(generate_argument_entries)

# Only select necessary columns for the output
output_df = tool_arguments_df[['File Name', 'Criteria Type', 'Arguments Entries', 'Cost', 'Error']]

output_df.to_excel(output_excel_path, index=False)
print(f"Data has been written to {output_excel_path}")


Data has been written to output_data_20240727_102327.xlsx


In [7]:
df

,File Name,Inclusion Criteria,Exclusion Criteria
0,NCT04898894.txt,\n\n* Participants must have a diagnosis of AM...,\n\n* Must not be pregnant or breastfeeding. M...
1,NCT04915612.txt,\n\n* Pediatric patients with diagnosis of CD3...,\n\n* History of another primary invasive mali...
2,NCT05188170.txt,\n\n* Prior morphologically confirmed diagnosi...,\n\n* Received anticancer therapy (chemotherap...


In [10]:
for item in abstract_list:
    file_name, inclusion_criteria, exclusion_criteria = item
    for criteria_type, criteria_text in [('Inclusion', inclusion_criteria), ('Exclusion', exclusion_criteria)]:
        print("\n" + "="*50)
        print(f"Processing {criteria_type} criteria for file {file_name}...")
        print("="*50 + "\n")
        
        clean_text_inclusion = criteria_text.replace("  ", " ").replace("\n", "; ").replace(';',' ')
        clean_text_exclusion = criteria_text.replace("  ", " ").replace("\n", "; ").replace(';',' ')
        print(clean_text_inclusion)
        print(clean_text_exclusion)




Processing Inclusion criteria for file NCT04898894.txt...

\n\n* Participants must have a diagnosis of AML or ALAL and meet the criteria below:\n\n * Refractory leukemia, defined as persistent leukemia after at least two courses of induction chemotherapy, OR\n * Early relapsed leukemia, defined as the re-appearance of leukemia after the achievement of remission and within one year of diagnosis, OR\n * Relapsed leukemia that is refractory to at least one course of salvage therapy (i.e., therapy given after the relapse has occurred), OR\n * Relapsed leukemia following HCT, OR\n * Second or greater relapse\n * Patients with late first relapses, defined as the re-appearance of leukemia after the achievement of remission and greater than one year of diagnosis, may be enrolled in the dose expansion portion of the study after safety data from the dose escalation portion is available.\n\nPatients must have ≥ 5% blasts in the bone marrow as assessed by morphology or flow cytometry. However, if

In [ ]:
for criteria_type, criteria_text in [('Inclusion', inclusion_criteria), ('Exclusion', exclusion_criteria)]:
    print("\n" + "="*50)
    print(f"Processing {criteria_type} criteria for file {file_name}...")
    print("="*50 + "\n")
    
    clean_text_inclusion = criteria_text.replace("  ", " ").replace("\n", "; ").replace(';',' ')
    clean_text_exclusion = criteria_text.replace("  ", " ").replace("\n", "; ").replace(';',' ')
